# MRI II — Oral Exam: EPI Sequence Assembly

**Time limit: 30 minutes**
 
All building blocks are already constructed for you below, with **two exceptions**: the phase-encoding blip gradient, and the readout-loop assembly. Both are marked as tasks. Once correct, the rest of the notebook (timing check, simulation, reconstruction) will run and produce an image — that image is your feedback.

In [ ]:
%matplotlib widget

import matplotlib.pyplot as plt
import MRzeroCore as mr0
import numpy as np
import pypulseq as pp
import torch

In [ ]:
seq_filename = "epi_exam.seq"

fov_x, fov_y = 256e-3, 256e-3
n_x, n_y = 128, 128
R = 1  # single-shot, fully sampled -- kept fixed for the exam

flip_angle_deg = 90
te = 25e-3

torch.set_default_device("cpu")

## System and Sequence Object

`system` holds the scanner hardware limits. `seq` is the container you add blocks (RF pulses, gradients, delays, ADC windows) to, in chronological order. Objects passed together to a single `seq.add_block()` call play *simultaneously*.

In [ ]:
system = pp.Opts(
    max_grad=28,
    grad_unit="mT/m",
    max_slew=150,
    slew_unit="T/m/s",
)

seq = pp.Sequence(system)

## Building Blocks (given)

- `rf` — excitation pulse
- `gx` — readout gradient, forward direction (+$k_x$)
- `gx_rev` — readout gradient, reversed direction (−$k_x$)
- `gx_pre` — moves to $-k_{x,\text{max}}$ before the first line
- `gy_pre` — moves to $-k_{y,\text{max}}$ before the first line
- `adc` — readout window (used with both `gx` and `gx_rev`)
- `gx_spoiler` — final spoiler after the whole readout train
`gy_blip` — the small phase-encoding step between consecutive lines — is **not** given; you will build it in Part 1 of the task below.

In [ ]:
delta_kx = 1 / fov_x
delta_ky = 1 / fov_y
raster_eps = 1e-12
flat_time = 1.6e-3

rf = pp.make_sinc_pulse(
    flip_angle=np.deg2rad(flip_angle_deg),
    system=system,
    use="excitation",
)

gx = pp.make_trapezoid(
    channel="x",
    flat_area=n_x * delta_kx,
    flat_time=flat_time,
    system=system,
)

gx_rev = pp.make_trapezoid(
    channel="x",
    flat_area=-n_x * delta_kx,
    flat_time=flat_time,
    system=system,
)

gx_pre = pp.make_trapezoid(
    channel="x",
    area=-gx.area / 2,
    system=system,
)

gy_pre = pp.make_trapezoid(
    channel="y",
    area=-n_y * delta_ky / 2,
    system=system,
)

adc = pp.make_adc(
    num_samples=n_x,
    duration=flat_time,
    delay=gx.rise_time,
    system=system,
)

gx_spoiler = pp.make_trapezoid(
    channel="x",
    area=4 * gx.area,
    system=system,
)

## Task: Assemble the EPI Sequence
 
In EPI, the **entire** k-space is acquired after a single RF excitation. Lines alternate direction, with a small blip gradient stepping k-space up between them. Since the blip's duration feeds directly into the TE calculation below, you'll build it first, then assemble the full readout loop afterward.
 
### Part 1 — Blip Gradient
 
Between consecutive readout lines, `gy_blip` must step $k_y$ by exactly one line — no more, no less — so that after `n_y` blips the full $k_y$ range has been traversed.
 
Build `gy_blip`: `channel="y"`, with the area required to step $k_y$ by one line (accounting for the acceleration factor `R`), using `system=system`.

In [ ]:
# START: SAMPLE SOLUTION

gy_blip = pp.make_trapezoid(
    channel="y",
    area=R * delta_ky,
    system=system,
)

# END: SAMPLE SOLUTION

## Timing (given)
 
`te_delay` is computed so the center of the readout train falls at the requested TE (a warning is printed if `te` is shorter than the achievable minimum). This depends on your `gy_blip` from Part 1 above.

In [ ]:
te_min = (
    pp.calc_duration(rf)
    - pp.calc_rf_center(rf)[0]
    + pp.calc_duration(gx_pre)
    + n_y / (2 * R) * (pp.calc_duration(gx) + pp.calc_duration(gy_blip))
)

te_delay = (
    np.ceil((te - te_min - raster_eps) / seq.grad_raster_time) * seq.grad_raster_time
)

if te_delay <= 0:
    print(
        f"Echo time {te} shorter than minimal echo time {te_min:.4f} s -> using {te_min:.4f} s"
    )
    te_delay = 0
    te = te_min

### Part 2 — Sequence Assembly
 
**Structure:**
1. Add the RF pulse.
2. Add `gx_pre` and `gy_pre` together (positions k-space at $(-k_{x,\text{max}}, -k_{y,\text{max}})$).
3. Add the TE delay.
4. Loop `n_y // 2` times, each iteration acquiring **two** lines:
   - add `gx` and `adc` together (forward line)
   - add `gy_blip`
   - add `gx_rev` and `adc` together (reversed line)
   - add `gy_blip`
5. Add `gx_spoiler`.

In [ ]:
# START: SAMPLE SOLUTION

seq.add_block(rf)

seq.add_block(gx_pre, gy_pre)

seq.add_block(pp.make_delay(te_delay))

for index in range(n_y // (2 * R)):

    seq.add_block(gx, adc)

    seq.add_block(gy_blip)

    seq.add_block(gx_rev, adc)

    seq.add_block(gy_blip)

seq.add_block(gx_spoiler)

# END: SAMPLE SOLUTION

## Verification, Simulation, Reconstruction (given)

In [ ]:
ok, error_report = seq.check_timing()

if ok:
    print("Timing check passed successfully")
else:
    print("Timing check failed. Error listing follows:")
    [print(e) for e in error_report]

plt.figure()
seq.plot()

seq.write(seq_filename, v141_compat=True)

In [ ]:
phantom = mr0.util.load_phantom(size=(n_x, n_y))

plt.figure()
phantom.plot()

mr0_seq = mr0.Sequence.import_file(seq_filename)

phantom.B0 = 0 * phantom.B0
phantom.T2dash = 1e6 + phantom.T2dash

data = phantom.build()

graph = mr0.compute_graph(mr0_seq, data)
signal = mr0.execute_graph(graph, mr0_seq, data, print_progress=True)

## Reconstruction (given)

EPI's critical difference from Cartesian GRE: **every other line was acquired in reverse**. Before applying the FFT, the odd-indexed lines must be flipped back into consistent order.

In [ ]:
signal = signal.numpy()
kspace = signal.reshape((n_y, n_x))

# undo the reversed-readout lines
kspace[1::2, :] = kspace[1::2, ::-1]

# Necessary to fix orientation
kspace = kspace[::-1, ::-1]

image = np.fft.fftshift(np.fft.ifft2(np.fft.ifftshift(kspace)))

plt.figure()
plt.imshow(np.abs(image), cmap="gray")
plt.title("Reconstructed EPI")
plt.show()